# 09 - Maintenance strategies and risk-based reasoning

## Learning objectives

- Distinguish corrective, preventive, CBM, PdM and RBM approaches
- Use TBM and UBM terminology correctly
- Interpret condition indicators and a simple RUL estimate
- Compare expected-cost maintenance options


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Maintenance strategies

- Corrective: after failure.
- Preventive TBM: time-based maintenance.
- Preventive UBM: usage-based maintenance.
- CBM: based on measured condition.
- PdM: based on predicted condition or remaining useful life.
- RBM: combines risk, consequences, costs and available opportunities.

These approaches are not automatically better or worse than one another. Select according to the information available and the decision objective.


## 2. Preventive interval: the first trade-off

For constant $\lambda$, $F(T)=1-e^{-\lambda T}$. A simple teaching cost is $C(T)=C_{PM}+F(T)C_F$. This is not a full renewal-reward or cost-per-unit-time optimization.


In [ ]:
lam=1/3000; C_PM=150_000; C_F=2_000_000
T=np.linspace(50,5000,200); F=1-np.exp(-lam*T); C=C_PM+F*C_F
plt.plot(T,C); plt.xlabel('preventive interval [h]'); plt.ylabel('simplified cost [HUF/cycle]'); plt.grid(alpha=.3); plt.show()


## 3. CBM: indicator and threshold

Generate a synthetic sensor-bias indicator. Crossing a threshold can initiate a condition-based action.


In [ ]:
rng=np.random.default_rng(4); t=np.arange(120); bias=np.maximum(0,.04*(t-40))+rng.normal(0,.15,len(t)); threshold=2.
alarm=bias>=threshold
plt.plot(t,bias); plt.axhline(threshold,ls='--'); plt.xlabel('time'); plt.ylabel('bias indicator'); plt.grid(alpha=.3); plt.show()
print('first crossing:',t[np.argmax(alarm)] if alarm.any() else None)


## 4. PdM: a simple RUL example

Fit a linear trend to the last 30 observations and estimate threshold-crossing time. Real prognostics need uncertainty treatment, drift detection, covariates and model diagnostics. A negative calculated RUL indicates that the fitted crossing is already in the past.


In [ ]:
w=30; slope,intercept=np.polyfit(t[-w:],bias[-w:],1); crossing=(threshold-intercept)/slope if slope>0 else np.inf; RUL=crossing-t[-1]
print('predicted crossing',crossing,'RUL',RUL)


## 5. RBM: maintain now or wait?

Simplified decision cost:

$$C_{expected}=C_{maintenance}+C_{production}+P(H)C_{consequence}.$$

Compare two options. Safety constraints and minimum protection requirements cannot simply be traded away through this monetary proxy.


In [ ]:
def expected(Cm,Cp,p,Cc): return Cm+Cp+p*Cc
opts=pd.DataFrame([
 {'option':'Maintain now','cost':expected(250_000,800_000,.001,50_000_000)},
 {'option':'Wait for opportunity','cost':expected(250_000,100_000,.02,50_000_000)}])
opts


## 6. Break-even probability

At which waiting-option hazard probability do the two expected costs become equal? Compare probabilities over the same exposure horizon.


In [ ]:
Cc=50_000_000; p_now=.001; prod_now=800_000; prod_window=100_000
p_break=(prod_now-prod_window+p_now*Cc)/Cc
print('break-even p_wait=',p_break)


In [ ]:
from safetycourse.maintenance import compare_maintenance_options
compare_maintenance_options([
 {'name':'Maintain now','maintenance_cost':250_000,'production_loss':800_000,'hazard_probability':.001,'consequence_cost':50_000_000},
 {'name':'Wait','maintenance_cost':250_000,'production_loss':100_000,'hazard_probability':.02,'consequence_cost':50_000_000}])


## Try it!

1. Double the consequence cost and recalculate the break-even value.
2. Reduce immediate production loss to HUF 300,000.
3. Specify the data needed to estimate p_wait rather than entering it manually.
4. Classify proof-test-based maintenance and explain why one label is not always sufficient.


## Summary

Maintenance strategy depends on information and objectives. Next, turn **sensor data into diagnostic evidence**, which can subsequently become a dynamic risk input.
